In [13]:
!pip install streamlit

!pip install pyngrok

In [14]:
%%writefile app.py
import streamlit as st

Overwriting app.py


In [ ]:
%%writefile app.py

import streamlit as st
from PIL import Image
import torch
from torchvision import models, transforms
import torch.nn as nn

# --- Page Config ---
st.set_page_config(page_title="SolarGuard - Panel Classifier", layout="wide")

# --- Load Model ---
@st.cache_resource
def load_model():
    model = models.resnet18(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 6)
    model.load_state_dict(torch.load("/content/solarr_classification_model (1).pth", map_location=torch.device("cpu")))
    model.eval()
    return model

model = load_model()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- Preprocessing ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class_names = ['Bird Drop', 'Clean', 'Dusty', 'Electrical Damage', 'Physical Damage', 'Snow Covered']

# --- Sidebar Navigation ---
st.sidebar.title("🔍 Navigation")
page = st.sidebar.radio("Go to", ["🏠 Home", "📄 Model Details", "📸 Solar Classifier"])

# --- Home Page ---
if page == "🏠 Home":
    st.title("🔆 Welcome to SolarGuard")
    st.markdown("""
    SolarGuard is an AI-powered tool to **automatically detect solar panel conditions** from uploaded images.

    ### 🌟 What It Does:
    - 📸 Classifies panel images as:
        - Dusty
        - Bird Dropping
        - Electrical/Physical Damage
        - Snow Covered
    - ⚡ Built with deep learning (ResNet18)

    ---
    👉 Use the '📸 Panel Classifier' to test the model!
    """)

# --- Details Page ---
elif page == "📄 Details":
    st.title("📊 Model & App Details")
    st.markdown("""
    ### 📦 Model: ResNet-18
    - Modified final layer to classify into 6 solar panel conditions.
    - Trained on labeled solar panel image dataset.
    - Efficient and accurate even on edge devices.

    ### 🛠 Technologies Used:
    - PyTorch for Deep Learning
    - Streamlit for Web App
    - PIL & TorchVision for Image Processing

    ### 📂 Classes:
    1. Bird Drop
    2. Clean
    3. Dusty
    4. Electrical Damage
    5. Physical Damage
    6. Snow Covered

    ---
    Built to help solar maintenance teams quickly identify panels needing attention. ⚡
    """)

# --- Panel Classifier Page ---
elif page == "📸 Panel Classifier":
    st.title("📸 Solar Panel Image  Classifier")

    uploaded_file = st.file_uploader("Upload a solar panel image (JPG/PNG)", type=["jpg", "jpeg", "png"])

    if uploaded_file is not None:
        image = Image.open(uploaded_file).convert('RGB')
        st.image(image, caption="📷 Uploaded Image", use_container_width=True)

        # Prediction
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            _, pred = torch.max(output, 1)
            predicted_class = class_names[pred.item()]
            confidence = torch.nn.functional.softmax(output, dim=1)[0][pred.item()].item()

        st.success(f"🧾 **Predicted Condition:** `{predicted_class}`")
        st.info(f"🔍 **Confidence:** `{confidence:.2%}`")


In [19]:
!npm install localtunnel

⠙⠹⠸⠼⠴
up to date, audited 23 packages in 859ms
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠦

In [22]:
!streamlit run /content/app.py &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

35.185.194.191
⠙your url is: https://gold-radios-learn.loca.lt
